In [ ]:
import cv2
from ultralytics import YOLO
import numpy as np
from pathlib import Path
import json
import time

# Define model and processing class
class PoseExtractor:
    def __init__(self, model_path='yolo11n-pose.pt'):
        print(f"Loading YOLO model: {model_path}")
        self.model = YOLO(model_path)
        print("Model loaded successfully!\n")
        
    def extract_keypoints(self, vid_path):
        cap = cv2.VideoCapture(str(vid_path))
        
        if not cap.isOpened():
            raise ValueError(f"Cannot open video: {vid_path}")
        
        # Get video properties
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        keypoints_sequence = []
        frame_idx = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Run YOLO pose estimation
            results = self.model(frame, verbose=False)
            
            # Extract keypoints (assuming single person in frame)
            if len(results[0].keypoints) > 0:
                # Get first detected person's keypoints
                # Shape: [17, 3] where 3 is (x, y, confidence)
                kpts = results[0].keypoints.data[0].cpu().numpy()
            else:
                # No person detected - use zeros
                kpts = np.zeros((17, 3))
            
            keypoints_sequence.append(kpts)
            frame_idx += 1
            
            # Progress indicator
            if frame_idx % 30 == 0:
                print(f"    Processed {frame_idx}/{total_frames} frames", end='\r')
        
        cap.release()
        
        # Convert to numpy array: [num_frames, 17, 3]
        keypoints_array = np.array(keypoints_sequence)
        
        return keypoints_array, fps, width, height

# MAIN
input_dir = "vids"
output_dir = "data/extracted_keypoints"
input_path = Path(input_dir)
output_path = Path(output_dir)

if not input_path.exists():
    raise ValueError(f"Input directory does not exist: {input_path}")

# Initialise pose extractor
extractor = PoseExtractor()

# Find all videos recursively
vid_extensions = ['.mp4', '.avi', '.mov', '.MP4', '.AVI', '.MOV']
vid_files = []
for ext in vid_extensions:
    vid_files.extend(input_path.rglob(f'*{ext}'))

if len(vid_files) == 0:
    print(f"WARNING: No video files found in {input_path}")
    exit()

print(f"Found {len(vid_files)} videos to process")

response = input("Continue? (y/n): ")
if response.lower() != 'y':
    print("Aborted.")
    exit()

print("\nStarting pose extraction...\n")

metadata = {}
start_time = time.time()

for i, vid_path in enumerate(vid_files, 1):
    # Get relative path to preserve folder structure
    rel_path = vid_path.relative_to(input_path)
    
    print(f"[{i}/{len(vid_files)}] {rel_path}")
    
    try:
        # Extract keypoints
        keypoints, fps, width, height = extractor.extract_keypoints(vid_path)
        
        # Create output path preserving folder structure
        output_file = output_path / rel_path.parent / f"{rel_path.stem}.npy"
        
        # Create output directory if needed
        output_file.parent.mkdir(parents=True, exist_ok=True)
        
        # Save keypoints
        np.save(output_file, keypoints)
        
        # Store metadata
        metadata_key = str(rel_path.parent / rel_path.stem)
        metadata[metadata_key] = {
            'original_video': str(rel_path),
            'num_frames': len(keypoints),
            'fps': fps,
            'keypoints_file': str(output_file),
            'category': str(rel_path.parent), # Directory
            'keypoints_shape': list(keypoints.shape),
            'width': width,
            'height': height,
            'resolution': f"{width}x{height}"
        }
        
        print(f"\n  Saved {len(keypoints)} frames\n")
        
    except Exception as e:
        print(f"  ! ERROR: {e}\n")
        continue
    
    # Print progress every 100 videos
    if i % 100 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(vid_files) - i) * avg_time
        print(f"\n--- Progress: {i}/{len(vid_files)} ({i/len(vid_files)*100:.1f}%) ---")
        print(f"    Elapsed: {elapsed/3600:.1f}h, Estimated remaining: {remaining/3600:.1f}h\n")

# Save metadata
metadata_file = output_path / 'extraction_metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

total_time = time.time() - start_time

print("\n" + "=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)
print(f"Processed {len(metadata)}/{len(vid_files)} videos successfully")
print(f"Total time: {total_time/3600:.1f} hours")
print(f"Average per video: {total_time/len(metadata):.1f} seconds")
print(f"Metadata saved to {metadata_file}")
print(f"\nKeypoints saved to: {output_path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/jovyan/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading YOLO model: yolo11n-pose.pt
Model loaded successfully!

Found 5678 videos to process
This will take approximately 8-12 hours



Continue? (y/n):  y



Starting pose extraction...

[1/5678] no-fall/bed/nf_raw_b_2/B_N_46.mp4
    Processed 90/96 frames
  ✓ Saved 96 frames

[2/5678] no-fall/bed/nf_raw_b_2/B_N_76_resized.mp4
    Processed 90/90 frames
  ✓ Saved 90 frames

[3/5678] no-fall/bed/nf_raw_b_2/B_N_67_resized.mp4
    Processed 120/120 frames
  ✓ Saved 120 frames

[4/5678] no-fall/bed/nf_raw_b_2/B_N_99.mp4
    Processed 150/150 frames
  ✓ Saved 150 frames

[5/5678] no-fall/bed/nf_raw_b_2/B_N_62_resized.mp4
    Processed 30/48 frames
  ✓ Saved 48 frames

[6/5678] no-fall/bed/nf_raw_b_2/B_N_54_resized.mp4

  ✓ Saved 24 frames

[7/5678] no-fall/bed/nf_raw_b_2/B_N_17.mp4
    Processed 90/90 frames
  ✓ Saved 90 frames

[8/5678] no-fall/bed/nf_raw_b_2/B_N_01.mp4
    Processed 90/90 frames
  ✓ Saved 90 frames

[9/5678] no-fall/bed/nf_raw_b_2/B_N_63_resized.mp4
    Processed 30/48 frames
  ✓ Saved 48 frames

[10/5678] no-fall/bed/nf_raw_b_2/B_N_11.mp4
    Processed 90/90 frames
  ✓ Saved 90 frames

[11/5678] no-fall/bed/nf_raw_b_2/B_N_78